### [ELECT] Votação por Seção - Bronze Layer

Import libs and start spark context

In [6]:
import os, sys, subprocess, zipfile, tempfile
from datetime import datetime

sys.path.append(os.path.join(os.getcwd(), '/home/jovyan/work/src/core'))

import spark_session
import pyspark.sql.functions as F

In [7]:
spark = spark_session.build_spark()

Start variables

In [8]:
target_schema = 'brz_elect'
target_table  = 'votacao_secao'

state = 'RS'
years = [2022, 2018, 2014]

base_url = 'https://cdn.tse.jus.br/estatistica/sead/odsele/votacao_secao'

Create schema and ingest each year

In [9]:
spark_session.run_sql(f'CREATE SCHEMA IF NOT EXISTS {target_schema};')

CREATE SCHEMA


In [10]:
for year in years:
    url = f'{base_url}/votacao_secao_{year}_{state}.zip'
    print(f'Downloading {url}...')

    with tempfile.TemporaryDirectory() as tmpdir:
        zip_path = os.path.join(tmpdir, f'votacao_secao_{year}_{state}.zip')

        result = subprocess.run(
            [
                'curl', '-L', '--fail', '-o', zip_path,
                '-H', 'User-Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36',
                '-H', 'Accept-Language: pt-BR,pt;q=0.9',
                '-H', 'Accept-Encoding: gzip, deflate, br',
                '-H', 'Referer: https://dadosabertos.tse.jus.br/',
                url,
            ],
            capture_output=True, text=True
        )

        if result.returncode != 0:
            print(f'  curl stderr:\n{result.stderr}')
            raise Exception(f'Download failed for year {year} (exit code {result.returncode})')

        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(tmpdir)

        csv_files = [f for f in os.listdir(tmpdir) if f.lower().endswith('.csv')]
        print(f'  Found files: {csv_files}')

        for csv_file in csv_files:
            csv_path = os.path.join(tmpdir, csv_file)

            df = spark.read \
                      .option('header', 'true') \
                      .option('sep', ';') \
                      .option('encoding', 'latin1') \
                      .csv(csv_path)

            df = df.withColumn('ingestion_datetime', F.lit(datetime.now())) \
                   .withColumn('ingestion_year',     F.lit(year)) \
                   .withColumn('ingestion_state',    F.lit(state)) \
                   .withColumn('ingestion_file',     F.lit(url))

            row_count = df.count()
            spark_session.write_table(df, target_schema, f'{target_table}_{year}', 'overwrite')
            print(f'  Written {row_count} rows for year {year}')

  Found files: ['votacao_secao_2022_RS.csv']
  Written 3803616 rows for year 2022
  Found files: ['votacao_secao_2018_RS.csv']
  Written 3983832 rows for year 2018
  Found files: ['votacao_secao_2014_RS.csv']
  Written 3237703 rows for year 2014


In [11]:
spark.stop()